# PSF metrics gallery - synthetic PSFs with known aberrations

Each row below is a synthetic PSF built with `calmutils.restoration.synthetic_psf`
with a **known ground-truth aberration**, rendered next to its
`psf_metrics` / `second_moments` readout (xy cut, xz cut, y/x marginals in nm).
Use it to judge at a glance which metric responds to which aberration:

- 1D-marginal metrics (`fwhm_*`, `astigmatism`, `asymmetry_*`, `skew`,
  `sidelobe_*`, `ring_radius_*`) are **rotation-blind in the y-x plane**: a
  diagonal elongation only shows up as inflated `fwhm_y`/`fwhm_x` with
  `astigmatism ~ 0`.
- The **2nd-moment tensor** (`second_moments`, surfaced in `psf_metrics` as
  `orientation_deg` / `elongation` / `covariance_*`) is what captures the
  diagonal: `orientation_deg` = major-axis angle in the last-two-axes plane
  (45 = diagonal), `elongation` = sqrt(lambda_major/lambda_minor).

Note on donuts: a *pure* donut (zero center) is read by the sidelobe finder as
`sidelobe_ratio ~ 1` at the **diameter** (~2R), since it anchors on one side of
the ring.

Run top-to-bottom; no data needed.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from calmutils.restoration import (psf_metrics, second_moments, axis_profile,
                                   synthetic_psf)

PSZ = (100.0, 50.0, 50.0)   # nm, (z, y, x)
SHAPE = (13, 41, 41)
SIGMA = (2.0, 2.5, 2.5)     # px, (z, y, x)

# one row per known aberration: name, overrides, note
ROWS = [
    dict(name='clean Gaussian', kw={}, note='all metrics ~ 0'),
    dict(name='z-elongated', kw=dict(sigma=(4.0, 2.5, 2.5)), note='fwhm_z up'),
    dict(name='astigmatism (axis-aligned x)', kw=dict(astigmatism=0.4),
         note='astigmatism ~ 0.4, orientation 0'),
    dict(name='diagonal 45 deg, elong 1.8', kw=dict(orientation_deg=45.0, elongation=1.8),
         note='marginals round (astig ~ 0) but orientation 45 / elong 1.8'),
    dict(name='diagonal -30 deg, elong 1.5', kw=dict(orientation_deg=-30.0, elongation=1.5),
         note='orientation -30 / elong 1.5'),
    dict(name='coma (tail +x)', kw=dict(coma=1.0, coma_sigma=4.0, coma_shift=5.0),
         note='asymmetry_x / skew_x > 0'),
    dict(name='donut ring R=6 px', kw=dict(ring_radius=6.0),
         note='sidelobe ~ 1 at the diameter ~2R (see note above)'),
    dict(name='off-center peak (0, 1.5, -2)', kw=dict(offset=(0.0, 1.5, -2.0)),
         note='peak_offset_px = offset'),
    dict(name='truncated window', kw=dict(shape=(9, 15, 15), sigma=(2.0, 3.0, 3.0)),
         note='truncated=True, edge_level high, fwhm lower bounds'),
    dict(name='noisy + pedestal', kw=dict(background=5e-4, noise=200.0, seed=1),
         note='fwhm_* are pedestal-sensitive (inflated here); 2nd moments shell-subtract the pedestal and stay ~round (orientation ~0, elong ~1)'),
]

In [ ]:
for row in ROWS:
    kw = dict(row['kw'])
    shape = kw.pop('shape', SHAPE)
    sigma = kw.pop('sigma', SIGMA)
    psf = synthetic_psf(shape, sigma, **kw)
    m = psf_metrics(psf, psz=PSZ)
    sm = second_moments(psf, psz=PSZ)

    c = np.array(psf.shape) // 2
    fig, axs = plt.subplots(1, 4, figsize=(17, 4.2))
    axs[0].imshow(psf[c[0]], cmap='gray',
                  extent=[0, psf.shape[2] * PSZ[2], psf.shape[1] * PSZ[1], 0])
    axs[0].set_title('xy cut (center z)'); axs[0].set_xlabel('x (nm)'); axs[0].set_ylabel('y (nm)')
    axs[1].imshow(psf[:, c[1], :], cmap='gray',
                  extent=[0, psf.shape[2] * PSZ[2], psf.shape[0] * PSZ[0], 0])
    axs[1].set_title('xz cut (center y)'); axs[1].set_xlabel('x (nm)'); axs[1].set_ylabel('z (nm)')
    for ax_i, axn in enumerate((1, 2)):
        prof = axis_profile(psf, axn)
        pos = (np.arange(len(prof)) - c[axn]) * PSZ[axn]
        axs[2 + ax_i].plot(pos, prof)
        axs[2 + ax_i].set_title(f'marginal {"yx"[ax_i]} (FWHM {m["fwhm_phys"][axn]:.0f} nm)')
        axs[2 + ax_i].set_xlabel(f'{"yx"[ax_i]} (nm)')
    fig.suptitle(
        f"{row['name']}   [{row['note']}]\n"
        f"fwhm nm z,y,x = {np.round(m['fwhm_phys'], 0)} | astig {m['astigmatism']:+.2f} | "
        f"asym_x {m['asymmetry_px'][2]:+.2f} skew_x {m['skew'][2]:+.2f} | "
        f"sidelobe {m['sidelobe_ratio']:.2f} ring {m['ring_radius_px'][2]:.1f} px | "
        f"edge {m['edge_level']:.1e} trunc {m['truncated']} | peak_off {np.round(m['peak_offset_px'], 2)}\n"
        f"2nd moments: orientation {m['orientation_deg']:+.1f} deg | elongation {m['elongation']:.2f} | "
        f"sigma major/minor {m['sigma_major_phys']:.0f}/{m['sigma_minor_phys']:.0f} nm")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
# summary table: metric vs ground truth, one row per aberration
records = []
for row in ROWS:
    kw = dict(row['kw'])
    shape = kw.pop('shape', SHAPE)
    sigma = kw.pop('sigma', SIGMA)
    psf = synthetic_psf(shape, sigma, **kw)
    m = psf_metrics(psf, psz=PSZ)
    records.append({
        'case': row['name'],
        'fwhm_z_nm': round(m['fwhm_phys'][0], 1), 'fwhm_y_nm': round(m['fwhm_phys'][1], 1),
        'fwhm_x_nm': round(m['fwhm_phys'][2], 1),
        'astigmatism': round(m['astigmatism'], 3) if m['astigmatism'] is not None else None,
        'asym_x': round(m['asymmetry_px'][2], 3), 'skew_x': round(m['skew'][2], 3),
        'sidelobe': round(m['sidelobe_ratio'], 3),
        'ring_x_px': round(m['ring_radius_px'][2], 1) if np.isfinite(m['ring_radius_px'][2]) else None,
        'edge_level': round(m['edge_level'], 4), 'truncated': m['truncated'],
        'peak_off_yx': np.round(m['peak_offset_px'][1:], 2).tolist(),
        'orientation_deg': round(m['orientation_deg'], 1), 'elongation': round(m['elongation'], 2),
    })
pd.DataFrame(records).set_index('case')